# IEEE-CIS Fraud Detection — Decision Tree

Sections: **Cleaning → Feature Engineering → Feature Selection → Training**

In [1]:
!pip install dagshub mlflow scikit-learn pandas numpy -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/

In [2]:
import os, gc, warnings
import numpy as np
import pandas as pd
import mlflow, mlflow.sklearn
warnings.filterwarnings('ignore')

from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from mlflow.models.signature import infer_signature
from scipy.stats import randint, uniform
import logging
import warnings
import mlflow

warnings.filterwarnings("ignore") 
logging.getLogger("mlflow").setLevel(logging.ERROR)

def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.float32)
    return df


from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['MLFLOW_TRACKING_USERNAME'] = 'dgrig23'
os.environ['MLFLOW_TRACKING_PASSWORD'] = secrets.get_secret('DAGSHUB_TOKEN')

REPO = 'dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning'
mlflow.set_tracking_uri(f'https://dagshub.com/{REPO}.mlflow')
mlflow.set_experiment('DecisionTree_Training')
EXP_PREFIX = 'DecisionTree'
BASE = '/kaggle/input/competitions/ieee-fraud-detection/'
print('Ready.')


Ready.


## 1. Cleaning

In [3]:
## 1. Cleaning
train_trx = pd.read_csv(BASE + 'train_transaction.csv')
train_idn = pd.read_csv(BASE + 'train_identity.csv')
train_idn.columns = train_idn.columns.str.replace('-', '_')
train = train_trx.merge(train_idn, on='TransactionID', how='left')
del train_trx, train_idn; gc.collect()


train = reduce_mem_usage(train)

with mlflow.start_run(run_name=f'{EXP_PREFIX}_Cleaning'):
    HIGH_MISS = 0.9
    miss = train.isnull().mean()
    high_miss_cols = miss[miss > HIGH_MISS].index.tolist()
    train.drop(columns=high_miss_cols + ['TransactionID'], inplace=True, errors='ignore')
    y = train.pop('isFraud').copy()
    fraud_rate = y.mean()
    mlflow.log_params({'high_miss_threshold': HIGH_MISS, 'cols_dropped': len(high_miss_cols)})
    mlflow.log_metrics({'fraud_rate': round(float(fraud_rate), 4), 'cols_after': train.shape[1]})
    print(f'Fraud rate: {fraud_rate:.4f} | Columns: {train.shape[1]}')

Fraud rate: 0.0350 | Columns: 420
🏃 View run DecisionTree_Cleaning at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/3/runs/cb3bd48fec1240529633ead30f64acd3
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/3


## 2. Feature Engineering

In [4]:
with mlflow.start_run(run_name=f'{EXP_PREFIX}_Feature_Engineering'):
    cols_before = train.shape[1]
    
    if 'TransactionDT' in train.columns:
        train['hour'] = ((train['TransactionDT'] / 3600) % 24).astype(np.float32)
        train['day'] = ((train['TransactionDT'] / (3600 * 24)) % 7).astype(np.float32)
        train.drop(columns=['TransactionDT'], inplace=True)
    
    for col in ['P_emaildomain', 'R_emaildomain']:
        if col in train.columns:
            train[col + '_suffix'] = train[col].str.split('.').str[-1].fillna('unknown')
            train[col + '_domain'] = train[col].str.split('.').str[0].fillna('unknown')

    for g in ['card1', 'card4', 'addr1']:
        if g in train.columns:
            agg_map = train.groupby(g)['TransactionAmt'].agg(['mean', 'std']).fillna(0)
            train[f'{g}_amt_mean'] = train[g].map(agg_map['mean'])
            train[f'{g}_amt_std'] = train[g].map(agg_map['std'])
            train[f'{g}_amt_zscore'] = ((train['TransactionAmt'] - train[f'{g}_amt_mean']) / train[f'{g}_amt_std'].replace(0, 1)).clip(-5, 5)

    train = reduce_mem_usage(train)
    mlflow.log_metrics({'new_features': train.shape[1] - cols_before, 'total_features': train.shape[1]})
    print(f'Features: {cols_before} → {train.shape[1]}')

Features: 420 → 434
🏃 View run DecisionTree_Feature_Engineering at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/3/runs/6301d55f3d684f089ed297a4f8979585
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/3


## 3. Feature Selection

In [5]:
cat_cols=train.select_dtypes(include='object').columns.tolist(); le_store={}
for col in cat_cols:
    le=LabelEncoder(); train[col]=train[col].fillna('unknown').astype(str)
    le.fit(list(train[col].unique())+['unknown']); le_store[col]=le; train[col]=le.transform(train[col])
imputer=SimpleImputer(strategy='median'); num_cols=train.select_dtypes(include=[np.number]).columns.tolist()
train[num_cols]=imputer.fit_transform(train[num_cols])
print(f'Missing: {train.isnull().sum().sum()}')


Missing: 0


In [6]:
X_tr, X_va, y_tr, y_va = train_test_split(train, y, test_size=0.2, stratify=y, random_state=42)

def quick_eval(features):
    m=DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=42)
    m.fit(X_tr[features], y_tr)
    return roc_auc_score(y_va, m.predict_proba(X_va[features])[:,1])

with mlflow.start_run(run_name=f'{EXP_PREFIX}_Feature_Selection'):
    all_features=train.columns.tolist()
    auc_all=quick_eval(all_features)
    print(f'Strategy A – All ({len(all_features)}): AUC = {auc_all:.5f}')

    vt=VarianceThreshold(threshold=0.01); vt.fit(train)
    vt_features=train.columns[vt.get_support()].tolist()
    auc_vt=quick_eval(vt_features)
    print(f'Strategy B – VarianceThreshold ({len(vt_features)}): AUC = {auc_vt:.5f}')

    corr=train.corrwith(y).abs(); corr_features=corr[corr>=0.01].index.tolist()
    auc_corr=quick_eval(corr_features)
    print(f'Strategy C – Correlation ({len(corr_features)}): AUC = {auc_corr:.5f}')

    imp_clf=DecisionTreeClassifier(max_depth=8,class_weight='balanced',random_state=42)
    imp_clf.fit(X_tr,y_tr); imp=pd.Series(imp_clf.feature_importances_,index=train.columns)
    top_features=imp.nlargest(100).index.tolist()
    auc_top=quick_eval(top_features)
    print(f'Strategy D – Top-100 importance ({len(top_features)}): AUC = {auc_top:.5f}')

    best_strategy,best_auc,final_features=max(
        [('all',auc_all,all_features),('variance_threshold',auc_vt,vt_features),
         ('correlation',auc_corr,corr_features),('top100_importance',auc_top,top_features)],
        key=lambda x:x[1]
    )
    mlflow.log_params({'strategy_A':'all','strategy_B':'variance_threshold_0.01','strategy_C':'correlation_0.01','strategy_D':'dt_importance_top100','selected':best_strategy,'n_final':len(final_features)})
    mlflow.log_metrics({'auc_all':round(auc_all,5),'auc_vt':round(auc_vt,5),'auc_corr':round(auc_corr,5),'auc_top100':round(auc_top,5),'best_auc':round(best_auc,5)})
    print(f'→ Best: {best_strategy} | AUC: {best_auc:.5f}')


Strategy A – All (434): AUC = 0.85845
Strategy B – VarianceThreshold (409): AUC = 0.85872
Strategy C – Correlation (318): AUC = 0.85855
Strategy D – Top-100 importance (100): AUC = 0.85815
→ Best: variance_threshold | AUC: 0.85872
🏃 View run DecisionTree_Feature_Selection at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/3/runs/780ebe36a80645b391ea4498c532bad7
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/3


## 4. Training — Decision Tree

In [7]:
X=train[final_features].copy()
X_tr,X_va,y_tr,y_va=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

# Underfitting — stump
with mlflow.start_run(run_name=f'{EXP_PREFIX}_Underfit_Config'):
    m=DecisionTreeClassifier(max_depth=1,class_weight='balanced',random_state=42)
    m.fit(X_tr,y_tr)
    tr_auc=roc_auc_score(y_tr,m.predict_proba(X_tr)[:,1]); va_auc=roc_auc_score(y_va,m.predict_proba(X_va)[:,1])
    mlflow.log_params({'max_depth':1,'note':'depth1_stump_underfit'})
    mlflow.log_metrics({'train_auc':round(tr_auc,5),'val_auc':round(va_auc,5),'overfit_gap':round(tr_auc-va_auc,5)})
    print(f'Underfit — Train: {tr_auc:.5f} | Val: {va_auc:.5f}')

# Overfitting — unlimited depth
with mlflow.start_run(run_name=f'{EXP_PREFIX}_Overfit_Config'):
    m=DecisionTreeClassifier(max_depth=None,min_samples_leaf=1,class_weight='balanced',random_state=42)
    m.fit(X_tr,y_tr)
    tr_auc=roc_auc_score(y_tr,m.predict_proba(X_tr)[:,1]); va_auc=roc_auc_score(y_va,m.predict_proba(X_va)[:,1])
    mlflow.log_params({'max_depth':'None','min_samples_leaf':1,'note':'unlimited_depth_memorises_data'})
    mlflow.log_metrics({'train_auc':round(tr_auc,5),'val_auc':round(va_auc,5),'overfit_gap':round(tr_auc-va_auc,5)})
    print(f'Overfit  — Train: {tr_auc:.5f} | Val: {va_auc:.5f} | Gap: {tr_auc-va_auc:.5f}')


Underfit — Train: 0.62769 | Val: 0.62813
🏃 View run DecisionTree_Underfit_Config at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/3/runs/cf6576e67dd14d3b8b3291736b6dc446
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/3
Overfit  — Train: 1.00000 | Val: 0.76063 | Gap: 0.23937
🏃 View run DecisionTree_Overfit_Config at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/3/runs/b4446c9abe6c4f139df3ae55af796c11
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/3


In [8]:
# RandomizedSearch — 10 runs, 3-fold
with mlflow.start_run(run_name=f'{EXP_PREFIX}_RandomizedSearch') as run_rs:
    param_dist = {
        'max_depth':             [3,5,6,8,10,12,15,None],
        'min_samples_split':     randint(2, 100),
        'min_samples_leaf':      randint(1, 50),
        'max_features':          ['sqrt','log2',None,0.3,0.5],
        'criterion':             ['gini','entropy'],
        'min_impurity_decrease': uniform(0, 0.005),
    }
    base_clf=DecisionTreeClassifier(class_weight='balanced',random_state=42)
    skf3=StratifiedKFold(3,shuffle=True,random_state=42)
    rs=RandomizedSearchCV(base_clf,param_dist,n_iter=10,cv=skf3,scoring='roc_auc',
                          n_jobs=-1,return_train_score=True,random_state=42,verbose=1)
    rs.fit(X,y)

    for i,params in enumerate(rs.cv_results_['params']):
        label=f"d{params.get('max_depth','None')}_msl{params['min_samples_leaf']}_mf{params['max_features']}"
        with mlflow.start_run(run_name=f'DT_RS_{label}',nested=True):
            mlflow.log_params({str(k):str(v) for k,v in params.items()})
            mlflow.log_metrics({
                'cv_auc_mean':    round(float(rs.cv_results_['mean_test_score'][i]),5),
                'cv_auc_std':     round(float(rs.cv_results_['std_test_score'][i]),5),
                'train_auc_mean': round(float(rs.cv_results_['mean_train_score'][i]),5),
                'overfit_gap':    round(float(rs.cv_results_['mean_train_score'][i]-rs.cv_results_['mean_test_score'][i]),5),
            })

    best_params=rs.best_params_
    mlflow.log_params({'best_'+k: str(v) for k,v in best_params.items()})
    mlflow.log_metric('best_cv_auc', round(rs.best_score_,5))
    print(f'Best CV AUC: {rs.best_score_:.5f} | Params: {best_params}')


Fitting 3 folds for each of 10 candidates, totalling 30 fits
🏃 View run DT_RS_d8_msl8_mf0.5 at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/3/runs/131b570ab94b4e72ab7de5b4f46f4053
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/3
🏃 View run DT_RS_d15_msl11_mflog2 at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/3/runs/47e9805dc848494497990df526fa73f8
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/3
🏃 View run DT_RS_d10_msl3_mf0.3 at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/3/runs/06c3564008fe4c9a808f2262f65586fc
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/3
🏃 View run DT_RS_d5_msl2_mf0.3 at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-De

In [9]:
# Final Pipeline
class FraudPreprocessorDT(BaseEstimator, TransformerMixin):
    def __init__(self,miss_thresh=0.9):
        self.miss_thresh=miss_thresh; self.high_miss_cols_=[]; self.le_store_={}; self.imputer_=SimpleImputer(strategy='median'); self.features_=None
    def fit(self,X,y=None):
        X=X.copy(); X.columns=X.columns.str.replace('-','_')
        for c in ['TransactionID','isFraud']:
            if c in X.columns: X.drop(columns=[c],inplace=True)
        self.high_miss_cols_=X.columns[X.isnull().mean()>self.miss_thresh].tolist()
        X.drop(columns=self.high_miss_cols_,inplace=True,errors='ignore'); X=self._engineer(X)
        for col in X.select_dtypes(include='object').columns:
            le=LabelEncoder(); vals=X[col].fillna('unknown').astype(str)
            le.fit(list(vals.unique())+['unknown']); self.le_store_[col]=le; X[col]=le.transform(vals)
        num=X.select_dtypes(include=[np.number]).columns.tolist()
        self.imputer_.fit(X[num]); X[num]=self.imputer_.transform(X[num]); self.features_=X.columns.tolist(); return self
    def transform(self,X):
        X=X.copy(); X.columns=X.columns.str.replace('-','_')
        for c in ['TransactionID','isFraud']:
            if c in X.columns: X.drop(columns=[c],inplace=True)
        X.drop(columns=self.high_miss_cols_,inplace=True,errors='ignore'); X=self._engineer(X)
        for col in X.select_dtypes(include='object').columns:
            if col in self.le_store_:
                known=set(self.le_store_[col].classes_); X[col]=X[col].fillna('unknown').astype(str)
                X[col]=X[col].apply(lambda v: v if v in known else 'unknown'); X[col]=self.le_store_[col].transform(X[col])
        num=X.select_dtypes(include=[np.number]).columns.tolist(); X[num]=self.imputer_.transform(X[num])
        for f in self.features_:
            if f not in X.columns: X[f]=0
        return X[self.features_]
    def _engineer(self,df):
        if 'TransactionDT' in df.columns:
            df['hour']=((df['TransactionDT']/3600)%24).astype(np.float32)
            df['dayofweek']=((df['TransactionDT']/(3600*24))%7).astype(np.float32)
            df.drop(columns=['TransactionDT'],inplace=True)
        for col in ['P_emaildomain','R_emaildomain']:
            if col in df.columns:
                df[col+'_suffix']=df[col].apply(lambda x: x.split('.')[-1] if isinstance(x,str) else 'unknown')
        if 'P_emaildomain' in df.columns and 'R_emaildomain' in df.columns:
            df['email_match']=(df['P_emaildomain']==df['R_emaildomain']).astype(np.int8)
        if 'TransactionAmt' in df.columns:
            df['TransactionAmt_log']=np.log1p(df['TransactionAmt'])
            df['amt_is_round']=(df['TransactionAmt']%1==0).astype(np.int8)
        return df

print('Reloading raw data...')
raw_trx=pd.read_csv(BASE+'train_transaction.csv'); raw_idn=pd.read_csv(BASE+'train_identity.csv')
raw_train=raw_trx.merge(raw_idn,on='TransactionID',how='left'); y_raw=raw_train['isFraud'].copy()
del raw_trx, raw_idn; gc.collect()

with mlflow.start_run(run_name=f'{EXP_PREFIX}_Final_Model'):
    final_clf=DecisionTreeClassifier(class_weight='balanced',random_state=42,**best_params)
    final_pipeline=Pipeline([('preprocessor',FraudPreprocessorDT()),('classifier',final_clf)])
    final_pipeline.fit(raw_train,y_raw)
    mlflow.log_params({**{str(k):str(v) for k,v in best_params.items()},'miss_thresh':0.9})
    mlflow.log_metric('final_cv_auc', round(rs.best_score_,5))
    sig=infer_signature(raw_train.head(5),final_pipeline.predict_proba(raw_train.head(5))[:,1])
    mlflow.sklearn.log_model(final_pipeline,artifact_path='dt_pipeline',signature=sig,registered_model_name='DecisionTree_FraudDetection')
    print(f'Registered. CV AUC: {rs.best_score_:.5f}')


Reloading raw data...


Registered model 'DecisionTree_FraudDetection' already exists. Creating a new version of this model...
Created version '2' of model 'DecisionTree_FraudDetection'.


Registered. CV AUC: 0.85607
🏃 View run DecisionTree_Final_Model at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/3/runs/69e32bac09bc4ccaa88e239c10f460b1
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/3
